# NYISO Solar Forecasting: Physics-Informed Feature Engineering for Residual Learning

## Imports and Configuration

In [34]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [35]:
repo_root = Path.home() / "Documents" / "Coding" / "ML_NYISOSolarForecast"

data_root = repo_root / "data"
processed_dir = data_root / "processed"

merged_out = processed_dir / "03_merged_data.csv"

In [36]:
df = pd.read_csv(merged_out, low_memory=False)

In [37]:
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

df["time_stamp"] = pd.to_datetime(df["time_stamp"], utc=True, errors="coerce")

if "time" in df.columns:
    df["time"] = pd.to_datetime(df["time"], utc=True, errors="coerce")
    same_time_mask = (df["time"] == df["time_stamp"]) | (
        df["time"].isna() & df["time_stamp"].isna()
    )
    if bool(same_time_mask.all()):
        df = df.drop(columns=["time"])

numeric_cols = [
    "actual_mw",
    "forecast_mw",
    "capacity_mw",
    "temperature_2m",
    "surface_pressure",
    "cloud_cover",
    "windspeed_10m",
    "shortwave_radiation",
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["zone_name"] = df["zone_name"].astype(str).str.strip().str.upper()

## Time and Error Metrics Engineering

In [43]:
df["time_local"] = df["time_stamp"].dt.tz_convert("America/New_York")
df["date_local"] = df["time_local"].dt.date
df["year_from_ts"] = df["time_local"].dt.year
df["quarter_local"] = df["time_local"].dt.quarter
df["month_local"] = df["time_local"].dt.month
df["day_local"] = df["time_local"].dt.day
df["dayofweek_local"] = df["time_local"].dt.dayofweek
df["hour_local"] = df["time_local"].dt.hour
df["is_weekend"] = df["dayofweek_local"].isin([5, 6]).astype(int)
df["is_daylight_proxy"] = (df["shortwave_radiation"] > 0).astype(int)

In [39]:
df["forecast_error_mw"] = df["actual_mw"] - df["forecast_mw"]
df["absolute_error_mw"] = (df["actual_mw"] - df["forecast_mw"]).abs()
df["smape"] = np.where(
    (df["actual_mw"].abs() + df["forecast_mw"].abs()) > 0,
    200 * df["absolute_error_mw"] / (df["actual_mw"].abs() + df["forecast_mw"].abs()),
    np.nan
)

## SYSTEM-only Feature Engineering Table

In [40]:
df_system = (
    df[df["zone_name"] == "SYSTEM"]
    .copy()
    .sort_values("time_stamp")
    .reset_index(drop=True)
)

print("System Shape:", df_system.shape)
print("System Time Range:", df_system["time_stamp"].min(), "to", df_system["time_stamp"].max())
print("System Columns:", df_system.columns.tolist())

System Shape: (42455, 24)
System Time Range: 2020-11-17 05:00:00+00:00 to 2025-09-21 03:00:00+00:00
System Columns: ['time_stamp', 'zone_name', 'actual_mw', 'forecast_mw', 'capacity_mw', 'temperature_2m', 'surface_pressure', 'cloud_cover', 'windspeed_10m', 'shortwave_radiation', 'year', 'time_local', 'date_local', 'year_from_ts', 'quarter_local', 'month_local', 'day_local', 'dayofweek_local', 'hour_local', 'is_weekend', 'is_daylight_proxy', 'forecast_error_mw', 'absolute_error_mw', 'smape']


In [41]:
df_system["dayofyear_local"] = df_system["time_local"].dt.dayofyear
df_system["weekofyear_local"] = df_system["time_local"].dt.isocalendar().week.astype(int)

df_system["hour_sin"] = np.sin(2 * np.pi * df_system["hour_local"] / 24)
df_system["hour_cos"] = np.cos(2 * np.pi * df_system["hour_local"] / 24)

df_system["month_sin"] = np.sin(2 * np.pi * df_system["month_local"] / 12)
df_system["month_cos"] = np.cos(2 * np.pi * df_system["month_local"] / 12)

df_system["dayofyear_sin"] = np.sin(2 * np.pi * df_system["dayofyear_local"] / 365.25)
df_system["dayofyear_cos"] = np.cos(2 * np.pi * df_system["dayofyear_local"] / 365.25)

df_system["dayofweek_sin"] = np.sin(2 * np.pi * df_system["dayofweek_local"] / 7)
df_system["dayofweek_cos"] = np.cos(2 * np.pi * df_system["dayofweek_local"] / 7)

In [42]:
df_system["is_morning_ramp"] = df_system["hour_local"].between(6, 9).astype(int)
df_system["is_midday"] = df_system["hour_local"].between(10, 14).astype(int)
df_system["is_evening_ramp"] = df_system["hour_local"].between(15, 18).astype(int)
df_system["is_night"] = (df_system["is_daylight_proxy"] == 0).astype(int)

In [ ]:
df_system["forecast_mw_log1p"] = np.log1p(df_system["forecast_mw"].clip(lower=0))
df_system["shortwave_radiation_log1p"] = np.log1p(df_system["shortwave_radiation"].clip(lower=0))

df_system["cloud_cover_frac"] = df_system["cloud_cover"] / 100.0

In [ ]:
df_system["capacity_ffill"] = df_system["capacity_mw"].ffill()
df_system["capacity_bfill"] = df_system["capacity_ffill"].bfill()
df_system["capacity_missing_flag"] = df_system["capacity_mw"].isna().astype(int)

df_system["forecast_capacity_ratio"] = np.where(
    df_system["capacity_bfill"].notna() & (df_system["capacity_bfill"] > 0),
    df_system["forecast_mw"] / df_system["capacity_bfill"],
    np.nan
)

df_system["actual_capacity_ratio"] = np.where(
    df_system["capacity_bfill"].notna() & (df_system["capacity_bfill"] > 0),
    df_system["actual_mw"] / df_system["capacity_bfill"],
    np.nan
)

## Physics-Informed Features Interactions

In [ ]:
df_system["forecast_x_hour_sin"] = df_system["forecast_mw"] * df_system["hour_sin"]
df_system["forecast_x_hour_cos"] = df_system["forecast_mw"] * df_system["hour_cos"]
df_system["forecast_x_daylight"] = df_system["forecast_mw"] * df_system["is_daylight_proxy"]

df_system["shortwave_x_cloud"] = df_system["shortwave_radiation"] * df_system["cloud_cover_frac"]
df_system["shortwave_x_temp"] = df_system["shortwave_radiation"] * df_system["temperature_2m"]
df_system["shortwave_x_wind"] = df_system["shortwave_radiation"] * df_system["windspeed_10m"]

df_system["forecast_x_shortwave"] = df_system["forecast_mw"] * df_system["shortwave_radiation"]
df_system["forecast_x_cloud"] = df_system["forecast_mw"] * df_system["cloud_cover_frac"]
df_system["forecast_x_temp"] = df_system["forecast_mw"] * df_system["temperature_2m"]

df_system["temp_x_wind"] = df_system["temperature_2m"] * df_system["windspeed_10m"]

df_system["cloud_x_hour_sin"] = df_system["cloud_cover_frac"] * df_system["hour_sin"]
df_system["cloud_x_hour_cos"] = df_system["cloud_cover_frac"] * df_system["hour_cos"]

## Lag and Rolling Features Maybe

In [ ]:
lag_hours = [1, 2, 3, 6, 12, 24]

for lag in lag_hours:
    df_system[f"forecast_mw_lag_{lag}"] = df_system["forecast_mw"].shift(lag)
    df_system[f"shortwave_radiation_lag_{lag}"] = df_system["shortwave_radiation"].shift(lag)
    df_system[f"cloud_cover_lag_{lag}"] = df_system["cloud_cover"].shift(lag)
    df_system[f"temperature_2m_lag_{lag}"] = df_system["temperature_2m"].shift(lag)
    df_system[f"windspeed_10m_lag_{lag}"] = df_system["windspeed_10m"].shift(lag)
    df_system[f"forecast_error_mw_lag_{lag}"] = df_system["forecast_error_mw"].shift(lag)

## Missing Summary

In [44]:
feature_missing_summary = (
    pd.DataFrame({
        "column": df_system.columns,
        "dtype": [str(df_system[c].dtype) for c in df_system.columns],
        "missing_count": [df_system[c].isna().sum() for c in df_system.columns],
    })
    .assign(missing_pct=lambda x: 100 * x["missing_count"] / len(df_system))
    .sort_values(["missing_count", "column"], ascending=[False, True])
)

feature_missing_summary.head(40)

,column,dtype,missing_count,missing_pct
4,capacity_mw,float64,23903,56.301967
23,smape,float64,18910,44.541279
22,absolute_error_mw,float64,1000,2.355435
21,forecast_error_mw,float64,1000,2.355435
2,actual_mw,float64,712,1.677070
3,forecast_mw,float64,288,0.678365
7,cloud_cover,int64,0,0.000000
12,date_local,object,0,0.000000
16,day_local,int32,0,0.000000
33,dayofweek_cos,float64,0,0.000000
